# Subgrids

A *subgrid* is a coarser grid whose points are a subset of a parent `Grid`'s points.
`SubGrid` is the API for building one.

This tutorial covers

1. [Creating a SubGrid](#Creating-a-SubGrid) -- the constructor and what it derives
2. [Defining Functions on it](#Defining-Functions-on-a-SubGrid)
3. [Writing Operators](#Writing-Operators) -- moving data between the two grids
4. [Finite differences](#Finite-differences-on-a-SubGrid)
5. [What SubGrid does not cover](#What-SubGrid-does-not-cover)
6. [The generated code](#The-generated-code) -- how the compiler lowers it

It covers **integer subsampling**, where subgrid points coincide exactly with parent
points, so no interpolation is involved.

## Creating a SubGrid

`SubGrid(parent, factor)` keeps every `factor`-th point of `parent` along each
`Dimension`, so subgrid point `i` sits at parent index `i*factor`.

In [1]:
import numpy as np
from devito import (CondEq, ConditionalDimension, Eq, Function, Grid, Operator,
                    SubGrid, TimeFunction)
from devito.ir.iet import FindNodes, Iteration

grid = Grid(shape=(12, 12), extent=(11., 11.))
x, y = grid.dimensions

sub = SubGrid(grid, factor=3)
sub

SubGrid[shape=(4, 4), factor=(3, 3)]

Everything else is derived from the parent. `shape` is `ceil(n/factor)` per axis:

In [2]:
print('parent  shape', grid.shape, ' subgrid shape', sub.shape)

parent  shape (12, 12)  subgrid shape (4, 4)


`spacing` is scaled by the factor, while `extent` and `origin` are inherited --
the subgrid spans the same physical domain with fewer points.

In [3]:
print('spacing  parent', tuple(float(v) for v in grid.spacing),
      ' subgrid', tuple(float(v) for v in sub.spacing))
print('extent   parent', tuple(float(v) for v in grid.extent),
      ' subgrid', tuple(float(v) for v in sub.extent))
print('origin   parent', tuple(float(v) for v in grid.origin),
      ' subgrid', tuple(float(v) for v in sub.origin))

spacing  parent (1.0, 1.0)  subgrid (3.0, 3.0)
extent   parent (11.0, 11.0)  subgrid (9.0, 9.0)
origin   parent (0.0, 0.0)  subgrid (0.0, 0.0)


`parent` and `factor` are the two things `SubGrid` adds over `Grid`:

In [4]:
sub.parent is grid, sub.factor

(True, (3, 3))

Its `dimensions` are `ConditionalDimension`s derived from the parent's. These are
what carry the subsampling factor into the generated code.

In [5]:
sub.dimensions, [d.parent for d in sub.dimensions]

((xs, ys), [x, y])

Pass `name` to control how they are named -- useful when building several
SubGrids off one parent.

In [6]:
SubGrid(grid, factor=2, name='c').dimensions

(xc, yc)

A factor per `Dimension` subsamples each axis independently, and `1` leaves an
axis untouched:

In [7]:
mixed = SubGrid(grid, factor=(3, 1))
print('shape     ', mixed.shape)
print('dimensions', mixed.dimensions)        # `y` passed through unchanged

shape      (4, 12)
dimensions (xs, y)


Being a `Grid` subclass, a SubGrid also carries the usual `Grid` attributes:

In [8]:
sub.dim, sub.dtype, sub.time_dim, sub.stepping_dim

(2, numpy.float32, time, t)

### What is rejected

An integer factor keeps subgrid points *on* parent points, which is plain
subsampling. A non-integer factor places them in between; that is also supported,
through interpolation, and is covered further down. What is rejected is a factor
that is not positive, or the wrong number of them.

In [9]:
for bad in (0, -1, (2, 2, 2)):
    try:
        SubGrid(grid, factor=bad)
    except ValueError as e:
        print(f'factor={bad!r:10} -> {e}')

factor=0          -> Subsampling factor must be > 0, got `0`
factor=-1         -> Subsampling factor must be > 0, got `-1`
factor=(2, 2, 2)  -> Expected 1 or 2 factors for a 2D parent Grid, got 3


Nor can a SubGrid be subsampled again; combine the factors off the original
parent instead.

In [10]:
try:
    SubGrid(sub, factor=2)
except ValueError as e:
    print(e)

Cannot subsample a SubGrid. `2` applied to a SubGrid of factor `(3, 3)` is the same as factor `(6, 6)` applied to its parent Grid directly, so use `SubGrid(parent.parent, factor=(6, 6))`


## Defining Functions on a SubGrid

This is the main convenience: a `Function` on a SubGrid needs neither `shape` nor
`dimensions`, both are taken from the grid -- exactly as for a normal `Grid`.

In [11]:
u = Function(name='u', grid=grid)                      # on the parent
usub = Function(name='usub', grid=sub, space_order=0)  # on the subgrid

print('u    ', u.shape, u.dimensions)
print('usub ', usub.shape, usub.dimensions)

u     (12, 12) (x, y)
usub  (4, 4) (xs, ys)


`TimeFunction` works too, and steps over the *parent's* time Dimension -- a
SubGrid coarsens space only.

In [12]:
usave = TimeFunction(name='usave', grid=sub, space_order=0)
usave.dimensions, usave.dimensions[0] is grid.stepping_dim

((t, xs, ys), True)

### Equivalent, by hand

`SubGrid` is a convenience over `ConditionalDimension`, which can still be used
directly. The manual form is equivalent but requires naming each Dimension and
computing the shape yourself.

In [13]:
xs_manual = ConditionalDimension(name='xsm', parent=x, factor=3)
ys_manual = ConditionalDimension(name='ysm', parent=y, factor=3)
f_manual = Function(name='f_manual', grid=grid,
                    dimensions=(xs_manual, ys_manual),
                    shape=(4, 4), space_order=0)        # shape by hand

f_manual.shape == usub.shape

True

## Writing Operators

An `Eq` between a parent Function and a SubGrid Function is a strided copy: Devito
matches the subgrid points to their parent counterparts.

In [14]:
# NBVAL_IGNORE_OUTPUT
u.data[:] = np.arange(144).reshape(12, 12)
usub.data[:] = 0.

Operator(Eq(usub, u)).apply()

assert np.array_equal(usub.data, u.data[::3, ::3])
usub.data

Operator `Kernel` ran in 0.01 s


Data([[  0.,   3.,   6.,   9.],
      [ 36.,  39.,  42.,  45.],
      [ 72.,  75.,  78.,  81.],
      [108., 111., 114., 117.]], dtype=float32)

Each row and column of `usub` comes from every third row and column of `u`.

The reverse direction is a choice, because the parent has points the subgrid does not.
`Eq(u, usub)` is evaluated at every point of `u` and interpolates the ones in between
(shown further down); passing `subdomain=sub` instead iterates the subgrid's points
alone, scattering onto them and leaving the others untouched.

In [15]:
# NBVAL_IGNORE_OUTPUT
u.data[:] = -1.
usub.data[:] = 7.

Operator(Eq(u, usub, subdomain=sub)).apply()      # scatter

assert np.all(u.data[::3, ::3] == 7.)
assert np.all(u.data[1::3, :] == -1.)      # untouched rows
u.data[:4, :4]

Operator `Kernel` ran in 0.01 s


Data([[ 7., -1., -1.,  7.],
      [-1., -1., -1., -1.],
      [-1., -1., -1., -1.],
      [ 7., -1., -1.,  7.]], dtype=float32)

Restricting the iteration range maps onto the subgrid. With `x_m=2` and
`factor=3`, the first subgrid point at or after `x=2` is `x=3`, i.e. subgrid index 1,
so index 0 is never written.

In [16]:
# NBVAL_IGNORE_OUTPUT
u.data[:] = 1.
usub.data[:] = -1.                     # sentinel: never written

Operator(Eq(usub, u)).apply(x_m=2)

usub.data[:, 0]

Operator `Kernel` ran in 0.01 s


Data([-1.,  1.,  1.,  1.], dtype=float32)

Several SubGrids can appear in one Operator, each with its own factor.

In [17]:
# NBVAL_IGNORE_OUTPUT
sub_a = SubGrid(grid, factor=2, name='a')
sub_b = SubGrid(grid, factor=4, name='b')
fa = Function(name='fa', grid=sub_a, space_order=0)
fb = Function(name='fb', grid=sub_b, space_order=0)

u.data[:] = np.arange(144).reshape(12, 12)
Operator([Eq(fa, u), Eq(fb, u)]).apply()

assert np.array_equal(fa.data, u.data[::2, ::2])
assert np.array_equal(fb.data, u.data[::4, ::4])
fa.shape, fb.shape

Operator `Kernel` ran in 0.01 s


((6, 6), (3, 3))

## Finite differences on a SubGrid

Derivatives of a SubGrid Function are taken with respect to the *coarse* spacing,
since the Dimension's `spacing` is `factor*h`.

In [18]:
sub.dimensions[0].spacing, x.spacing

(3*h_x, h_x)

So the stencil steps `factor*h_x` and the weights scale accordingly:

In [19]:
fsub = Function(name='fsub', grid=sub, space_order=2)
fsub.dx.evaluate

-0.333333333*fsub(xs, ys)/h_x + 0.333333333*fsub(xs + 3*h_x, ys)/h_x

Checking it numerically on a linear field, whose derivative is exact:

In [20]:
# NBVAL_IGNORE_OUTPUT
grid_l = Grid(shape=(21, 21), extent=(20., 20.))
sub_l = SubGrid(grid_l, factor=2)

fl = Function(name='fl', grid=sub_l, space_order=2)
dfl = Function(name='dfl', grid=sub_l, space_order=2)

# fl = 3*x, sampled on the coarse grid
for i in range(fl.shape[0]):
    fl.data[i, :] = 3.*i*float(sub_l.spacing[0])

Operator(Eq(dfl, fl.dx)).apply()

assert np.allclose(dfl.data[1:-1, 1:-1], 3., atol=1e-4)
float(dfl.data[5, 5])

Operator `Kernel` ran in 0.01 s


3.0

## What SubGrid does not cover

Two related things are deliberately outside `SubGrid` and still want a
`ConditionalDimension` built by hand.

**Subsampling in time.** A SubGrid coarsens space, so saving a wavefield every
`factor` timesteps uses a `ConditionalDimension` on the time Dimension.

In [21]:
# NBVAL_IGNORE_OUTPUT
nt, factor = 9, 4
grid_t = Grid(shape=(4, 4))
ts = ConditionalDimension(name='ts', parent=grid_t.time_dim, factor=factor)

v = TimeFunction(name='v', grid=grid_t)
vsave = TimeFunction(name='vsave', grid=grid_t, time_dim=ts,
                     save=(nt + factor - 1)//factor)

Operator([Eq(v.forward, v + 1.), Eq(vsave, v)]).apply(time_M=nt-2)

[float(vsave.data[i, 0, 0]) for i in range(vsave.shape[0])]

Operator `Kernel` ran in 0.01 s


[0.0, 4.0, 0.0]

**Extra conditions.** `SubGrid` takes a factor, not an arbitrary predicate; for
`factor` *and* a `condition`, construct the `ConditionalDimension` directly.

In [22]:
xs_cond = ConditionalDimension(name='xsc', parent=x, factor=3,
                               condition=CondEq(x % 2, 0))
xs_cond.factor, xs_cond.condition

(3, Eq(Mod(x, 2), 0))

Generic subsampling -- subgrid points that fall *between* parent points -- **is**
covered, in the next section.

## The generated code

The rest of this tutorial shows how Devito lowers the above, which is useful when
reading or profiling the generated C.

Devito iterates the **subgrid** directly: one unit-stride loop per subsampled
`Dimension`, with parent accesses rescaled -- rather than walking every parent point
under an `if (x%factor == 0)` guard.

In [23]:
def show(op):
    """Print the Operator's loop nest."""
    print(op.body.body[-1])


def steps(op):
    """The loop increment of each Iteration, by Dimension name."""
    return {i.dim.name: i.limits[2] for i in FindNodes(Iteration).visit(op)}

### One loop per subsampled Dimension

In [24]:
# NBVAL_IGNORE_OUTPUT
op = Operator(Eq(usub, u))
show(op)

START(section0)
for (int xs = xs_m; xs <= xs_M; xs += 1)
{
  #pragma omp simd aligned(u,usub:64)
  for (int ys = ys_m; ys <= ys_M; ys += 1)
  {
    usub[xs][ys] = u[xs*xsf + 1][ys*ysf + 1];
  }
}
STOP(section0,timers)


Note what the loop iterates: `xs`, the subsampled `Dimension` itself, with a
**unit** increment, over its own bounds `xs_m .. xs_M`. There is no `if` in the nest.

- the subgrid access is simply `usub[xs]`;
- the parent access is rescaled to `u[xs*xsf + 1]`.

The unit stride is deliberate. A `x += factor` loop would also remove the guard, but a
non-unitary increment escapes the Lamport-style dependence analysis the compiler
relies on, so it is avoided.

The bounds need no adjusting either: `xs_m` and `xs_M` are `ConditionalDimension`'s own
runtime arguments, derived as `ceil(x_m/factor)` and `floor(x_M/factor)`.

In [25]:
steps(op)

{'xs': 1, 'ys': 1}

### The innermost Dimension

Subsampling the innermost (vectorized) `Dimension` is handled the same way. The
alternative -- a guard inside the SIMD loop -- would mean a predicated store on every
iteration.

In [26]:
# NBVAL_IGNORE_OUTPUT
sub_y = SubGrid(grid, factor=(1, 2))      # coarsen y only
usub_y = Function(name='usub_y', grid=sub_y, space_order=0)

op_inner = Operator(Eq(usub_y, u))
show(op_inner)

START(section0)
for (int x = x_m; x <= x_M; x += 1)
{
  #pragma omp simd aligned(u,usub_y:64)
  for (int ys = ys_m; ys <= ys_M; ys += 1)
  {
    usub_y[x][ys] = u[x + 1][ys*ysf + 1];
  }
}
STOP(section0,timers)


In [27]:
steps(op_inner)

{'x': 1, 'ys': 1}

The `y` loop is strided and the `#pragma omp simd` is retained.

In [28]:
# NBVAL_IGNORE_OUTPUT
u.data[:] = np.arange(144).reshape(12, 12)
Operator(Eq(usub_y, u)).apply()

assert np.array_equal(usub_y.data, u.data[:, ::2])
usub_y.data[:3]

Operator `Kernel` ran in 0.01 s


Data([[ 0.,  2.,  4.,  6.,  8., 10.],
      [12., 14., 16., 18., 20., 22.],
      [24., 26., 28., 30., 32., 34.]], dtype=float32)

### Several Dimensions at once

Each subsampled `Dimension` gets its own loop, independently.

In [29]:
# NBVAL_IGNORE_OUTPUT
sub_xy = SubGrid(grid, factor=(3, 2))
usub_xy = Function(name='usub_xy', grid=sub_xy, space_order=0)

op_both = Operator(Eq(usub_xy, u))
show(op_both)

START(section0)
for (int xs = xs_m; xs <= xs_M; xs += 1)
{
  #pragma omp simd aligned(u,usub_xy:64)
  for (int ys = ys_m; ys <= ys_M; ys += 1)
  {
    usub_xy[xs][ys] = u[xs*xsf + 1][ys*ysf + 1];
  }
}
STOP(section0,timers)


In [30]:
# NBVAL_IGNORE_OUTPUT
u.data[:] = np.arange(144).reshape(12, 12)
Operator(Eq(usub_xy, u)).apply()

assert np.array_equal(usub_xy.data, u.data[::3, ::2])
steps(op_both)

Operator `Kernel` ran in 0.01 s


{'xs': 1, 'ys': 1}

## When a guard is used instead

Striding is applied only where it provably preserves semantics. Where it does not,
Devito visits every point under an `if (x%factor == 0)` guard. Results are the same
either way; only the generated loop differs. Four cases, each with its code.

### 1. The loop nest is shared with an unguarded equation

`Eq(f, 1)` must run at **every** `x`, while `Eq(g, f)` runs every fourth. Both land in
one loop nest, so striding it would also stride `Eq(f, 1)`. The guard is kept.

In [31]:
# NBVAL_IGNORE_OUTPUT
grid1 = Grid(shape=(12,))
sub1 = SubGrid(grid1, factor=4)

f = Function(name='f', grid=grid1)
g = Function(name='g', grid=sub1, space_order=0)

op_shared = Operator([Eq(f, 1), Eq(g, f)])
show(op_shared)

START(section0)
for (int x = x_m; x <= x_M; x += 1)
{
  f[x + 1] = 1;
  if ((x)%(xsf) == 0)
  {
    g[x / xsf] = f[x + 1];
  }
}
STOP(section0,timers)


In [32]:
# NBVAL_IGNORE_OUTPUT
f.data[:] = 0.
g.data[:] = -1.
op_shared.apply()

# `f` written at every point, `g` only on the subgrid
assert np.all(f.data == 1.)
assert np.all(g.data == 1.)
f.data[:3], g.data[:]

Operator `Kernel` ran in 0.01 s


(Data([1., 1., 1.], dtype=float32), Data([1., 1., 1.], dtype=float32))

`f[x + 1] = 1` sits outside the `if`, `g[x / xs1f]` inside it -- which is exactly
what is required.

### 2. The subsampled Dimension is sequential

Striding skips points, which is only sound when no dependence is carried along the
`Dimension`. Here `g` reads its own previous subgrid value, so `x` is sequential.

In [33]:
# NBVAL_IGNORE_OUTPUT
grid2 = Grid(shape=(12, 12))
sub2 = SubGrid(grid2, factor=(3, 1))
xs3 = sub2.dimensions[0]

f2 = Function(name='f2', grid=grid2)
g2 = Function(name='g2', grid=sub2, space_order=0)

op_seq = Operator(Eq(g2, g2.subs(xs3, xs3-1) + f2))
show(op_seq)

START(section0)
for (int x = x_m; x <= x_M; x += 1)
{
  if ((x)%(xsf) == 0)
  {
    for (int y = y_m; y <= y_M; y += 1)
    {
      g2[x / xsf][y] = f2[x + 1][y + 1] + g2[(x / xsf) - 1][y];
    }
  }
}
STOP(section0,timers)


In [34]:
# NBVAL_IGNORE_OUTPUT
f2.data[:] = 1.
g2.data[:] = 0.
op_seq.apply()

# Each subgrid row accumulates the one before it; row 0 reads the halo (zero)
assert np.allclose(g2.data[:, 0], [0., 1., 2., 3.])
g2.data[:, 0]

Shrinking bounds (`x_m`, `x_M`); some `x` points will not be computed. Likely insufficient space_order for the derivatives.


/home/mloubout/DevitoCodes/devito/devito/operator/operator.py:718: DevitoWarning: Shrinking bounds (`x_m`, `x_M`); some `x` points will not be computed. Likely insufficient space_order for the derivatives.
  warn(f"Shrinking bounds (`{d.min_name}`, `{d.max_name}`); "
Shrinking bounds (`xs_m`, `xs_M`); some `xs` points will not be computed. Likely insufficient space_order for the derivatives.


/home/mloubout/DevitoCodes/devito/devito/operator/operator.py:718: DevitoWarning: Shrinking bounds (`xs_m`, `xs_M`); some `xs` points will not be computed. Likely insufficient space_order for the derivatives.
  warn(f"Shrinking bounds (`{d.min_name}`, `{d.max_name}`); "
Operator `Kernel` ran in 0.01 s


Data([0., 1., 2., 3.], dtype=float32)

Time subsampling is the everyday instance of this: the time loop is sequential, so a
saved-wavefield `Eq` keeps its guard. Note this one uses a `ConditionalDimension`
directly -- a `SubGrid` coarsens *space*, so it has no say over the time axis.

In [35]:
# NBVAL_IGNORE_OUTPUT
nt, factor = 9, 4
grid3 = Grid(shape=(4, 4))
ts = ConditionalDimension(name='ts', parent=grid3.time_dim, factor=factor)

v = TimeFunction(name='v', grid=grid3)
vsave = TimeFunction(name='vsave', grid=grid3, time_dim=ts,
                     save=(nt + factor - 1)//factor)

op_time = Operator([Eq(v.forward, v + 1.), Eq(vsave, v)])
show(op_time)

for (int time = time_m, t0 = (time)%(2), t1 = (time + 1)%(2); time <= time_M; time += 1, t0 = (time)%(2), t1 = (time + 1)%(2))
{
  START(section0)
  for (int x = x_m; x <= x_M; x += 1)
  {
    #pragma omp simd aligned(v:64)
    for (int y = y_m; y <= y_M; y += 1)
    {
      v[t1][x + 1][y + 1] = v[t0][x + 1][y + 1] + 1.0F;
    }
  }
  STOP(section0,timers)

  if ((time)%(tsf) == 0)
  {
    START(section1)
    for (int x = x_m; x <= x_M; x += 1)
    {
      #pragma omp simd aligned(v,vsave:64)
      for (int y = y_m; y <= y_M; y += 1)
      {
        vsave[time / tsf][x + 1][y + 1] = v[t0][x + 1][y + 1];
      }
    }
    STOP(section1,timers)
  }
}


The saved slices are correct -- time subsampling behaves exactly as it always has,
it simply does not take the strided form.

In [36]:
# NBVAL_IGNORE_OUTPUT
op_time.apply(time_M=nt-2)

nwritten = (nt - 2)//factor + 1
for i in range(nwritten):
    assert np.allclose(vsave.data[i], i*factor)

[float(vsave.data[i, 0, 0]) for i in range(vsave.shape[0])]

Operator `Kernel` ran in 0.01 s


[0.0, 4.0, 0.0]

### 3. A `condition` is attached as well

A `ConditionalDimension` may carry an arbitrary `condition` alongside a `factor`. The
combined test is no longer "every `factor`-th point", so it is evaluated pointwise.

In [37]:
# NBVAL_IGNORE_OUTPUT
# `SubGrid` does not expose `condition`, so this one is built by hand
xs4 = ConditionalDimension(name='xs4', parent=x, factor=3,
                           condition=CondEq(x % 2, 0))
w = Function(name='w', grid=grid, dimensions=(xs4, y), shape=(4, 12),
             space_order=0)

u.data[:] = 1.
w.data[:] = -1.
op_cond = Operator(Eq(w, u))
op_cond.apply()
show(op_cond)

Operator `Kernel` ran in 0.01 s


START(section0)
for (int x = x_m; x <= x_M; x += 1)
{
  if ((x & 1) == 0 && (x)%(xs4f) == 0)
  {
    #pragma omp simd aligned(u,w:64)
    for (int y = y_m; y <= y_M; y += 1)
    {
      w[x / xs4f][y] = u[x + 1][y + 1];
    }
  }
}
STOP(section0,timers)


### 4. Accesses are not affine in the subsampled Dimension

An indirectly indexed read (`u[idx[x, y], y]`) is not affine in `x`, so the points a
stride would skip cannot be shown to be unneeded.

In [38]:
# NBVAL_IGNORE_OUTPUT
idx = Function(name='idx', grid=grid, dtype=np.int32)
gi = Function(name='gi', grid=sub, space_order=0)

idx.data[:] = 0                      # read row 0 of `u` everywhere
u.data[:] = np.arange(144).reshape(12, 12)
op_indirect = Operator(Eq(gi, u[idx[x, y], y]))
op_indirect.apply()

assert np.array_equal(gi.data, u.data[0, ::3] * np.ones((4, 1)))
show(op_indirect)

Operator `Kernel` ran in 0.01 s


START(section0)
for (int x = x_m; x <= x_M; x += 1)
{
  if ((x)%(xsf) == 0)
  {
    #pragma omp simd aligned(gi,idx,u:64)
    for (int y = y_m; y <= y_M; y += 1)
    {
      if ((y)%(ysf) == 0)
      {
        int r0 = idx[x + 1][y + 1];
        gi[x / xsf][y / ysf] = u[r0 + 1][y + 1];
      }
    }
  }
}
STOP(section0,timers)


## Interpolated subgrids

A non-integer factor puts subgrid points *between* parent points, so reading such a
Function on the parent grid has to interpolate. This needs no new API: a SubGrid
Function used in an equation is evaluated **at the left-hand side's location**, exactly
as a staggered Function is.

In [39]:
grid_i = Grid(shape=(13, 13), extent=(12., 12.))
sub_i = SubGrid(grid_i, factor=1.5)

spacing = lambda g: tuple(float(s) for s in g.spacing)
print('parent shape ', grid_i.shape, ' spacing', spacing(grid_i))
print('subgrid shape', sub_i.shape, ' spacing', spacing(sub_i))
print('interpolated ', sub_i.is_interpolated)
print('dimensions   ', sub_i.dimensions)

parent shape  (13, 13)  spacing (1.0, 1.0)
subgrid shape (9, 9)  spacing (1.5, 1.5)
interpolated  True
dimensions    (xs, ys)


`dimensions` are now plain `SpaceDimension`s with their own spacing, not
`ConditionalDimension`s: there is no stride to exploit, because the points do not line
up with the parent's.

### Reading a subgrid Function on the parent

`Eq(u, vp)` reads `vp` at `u`'s points -- no wrapper and no flag. The relocation happens
during evaluation, exactly as a staggered Function's does, so `evaluate` already shows
the finished stencil:

In [40]:
u_i = Function(name='u_i', grid=grid_i, space_order=2)
vp_i = Function(name='vp_i', grid=sub_i, space_order=2)

eq = Eq(u_i, vp_i)
eq.evaluate

Eq(u_i(x, y), vp_i((IntDiv(2*x, 3)) + 1, (IntDiv(2*y, 3)) + 1)*w6c8dafcd[2*(Mod(x, 3)) + 1]*w6c8dafcd[2*(Mod(y, 3)) + 1] + vp_i((IntDiv(2*x, 3)) + 1, IntDiv(2*y, 3))*w6c8dafcd[2*(Mod(x, 3)) + 1]*w6c8dafcd[2*(Mod(y, 3))] + vp_i(IntDiv(2*x, 3), (IntDiv(2*y, 3)) + 1)*w6c8dafcd[2*(Mod(y, 3)) + 1]*w6c8dafcd[2*(Mod(x, 3))] + vp_i(IntDiv(2*x, 3), IntDiv(2*y, 3))*w6c8dafcd[2*(Mod(x, 3))]*w6c8dafcd[2*(Mod(y, 3))])

Each term reads `vp_i` at a base index derived from the parent's, weighted by a
coefficient that depends on the parent index only through `x % 3`. Nothing is deferred:
this is an ordinary expression, so the iteration space, the halo scheme and the data
space are all derived from plain indexed accesses.

### The generated code

The weights live in a small compile-time table, so the loops stay plain and unit-stride
-- no branch, no `floorf`, and only the parent's Dimensions are iterated.

In [41]:
# NBVAL_IGNORE_OUTPUT
op_i = Operator(eq)
print(op_i)

/* Devito generated code for Operator `Kernel` */

#define _POSIX_C_SOURCE 200809L
#define START(S) struct timeval start_ ## S , end_ ## S ; gettimeofday(&start_ ## S , NULL);
#define STOP(S,T) gettimeofday(&end_ ## S, NULL); T->S += (double)(end_ ## S .tv_sec-start_ ## S.tv_sec)+(double)(end_ ## S .tv_usec-start_ ## S .tv_usec)/1000000;

#include "stdlib.h"
#include "math.h"
#include "sys/time.h"
#include "xmmintrin.h"
#include "pmmintrin.h"

struct dataobj
{
  void *restrict data;
  int * size;
  unsigned long nbytes;
  unsigned long * npsize;
  unsigned long * dsize;
  int * hsize;
  int * hofs;
  int * oofs;
  void * dmap;
} ;

struct profiler
{
  double section0;
} ;


int Kernel(struct dataobj *restrict u_i_vec, struct dataobj *restrict vp_i_vec, const int x_M, const int x_m, const int y_M, const int y_m, struct profiler * timers)
{
  float w6c8dafcdi[6] __attribute__ ((aligned (64))) = {1.0F, 0.0F, 3.33333333333333e-1F, 6.66666666666667e-1F, 6.66666666666667e-1F, 3.3333333333333

Reading at parent point `x` evaluates at subgrid coordinate `x/factor`. For
`factor = p/q` that is `x*q/p`, so the base index is `(x*q)//p` and the fractional part
`((x*q) % p)/p` repeats with period `p` -- hence a table of `p` phases, two linear taps
each.

### Linear interpolation is exact on a linear field

The sharpest check available: any error in a base index or a weight shows up at once.

In [42]:
idx = np.meshgrid(*[np.arange(n) for n in sub_i.shape], indexing='ij')
vp_i.data[:] = (3.*idx[0] + 2.*idx[1]).astype(vp_i.dtype)
u_i.data[:] = 0.

op_i.apply()

pidx = np.meshgrid(*[np.arange(n) for n in grid_i.shape], indexing='ij')
exact = 3.*(pidx[0]/1.5) + 2.*(pidx[1]/1.5)

# Only points whose two taps stay inside the subgrid
inside = (pidx[0] <= int((sub_i.shape[0] - 1)*1.5) - 1) & \
         (pidx[1] <= int((sub_i.shape[1] - 1)*1.5) - 1)
print('max error:', np.abs(np.asarray(u_i.data)[inside] - exact[inside]).max())

Operator `Kernel` ran in 0.01 s


max error: 3.178914390389309e-06


### A factor below 1 refines

Nothing in the lowering assumes coarsening -- `factor=0.5` makes the subgrid *finer*
than its parent. Every parent point then lands exactly on a subgrid point, so there is
a single phase: no weight table is emitted at all and the read collapses to a direct
strided copy.

In [43]:
# NBVAL_IGNORE_OUTPUT
sub_f = SubGrid(grid_i, factor=0.5)
print('subgrid shape', sub_f.shape, 'vs parent', grid_i.shape)

vp_f = Function(name='vp_f', grid=sub_f, space_order=2)
print(Operator(Eq(u_i, vp_f)))

subgrid shape (25, 25) vs parent (13, 13)
/* Devito generated code for Operator `Kernel` */

#define _POSIX_C_SOURCE 200809L
#define START(S) struct timeval start_ ## S , end_ ## S ; gettimeofday(&start_ ## S , NULL);
#define STOP(S,T) gettimeofday(&end_ ## S, NULL); T->S += (double)(end_ ## S .tv_sec-start_ ## S.tv_sec)+(double)(end_ ## S .tv_usec-start_ ## S .tv_usec)/1000000;

#include "stdlib.h"
#include "math.h"
#include "sys/time.h"
#include "xmmintrin.h"
#include "pmmintrin.h"

struct dataobj
{
  void *restrict data;
  int * size;
  unsigned long nbytes;
  unsigned long * npsize;
  unsigned long * dsize;
  int * hsize;
  int * hofs;
  int * oofs;
  void * dmap;
} ;

struct profiler
{
  double section0;
} ;


int Kernel(struct dataobj *restrict u_i_vec, struct dataobj *restrict vp_f_vec, const int x_M, const int x_m, const int y_M, const int y_m, struct profiler * timers)
{
  float (*restrict u_i)[u_i_vec->size[1]] __attribute__ ((aligned (64))) = (float (*)[u_i_vec->size[1]]) u_

### Per-axis factors

Each axis carries its own factor. A factor of exactly `1` passes the parent's own
Dimension straight through, so that axis is neither strided nor interpolated.

Note that *within* an interpolated SubGrid even an integer axis interpolates. With
`factor=2`, the parent point `x=1` lies halfway between two subgrid points, so reading
it genuinely needs both taps -- unlike the pure subsampling case, where the subgrid is
written and only its own points are visited.

In [44]:
# NBVAL_IGNORE_OUTPUT
grid_m = Grid(shape=(13, 13, 13), extent=(12.,)*3)
sub_m = SubGrid(grid_m, factor=(2, 1.5, 1.0))
print('dimensions', sub_m.dimensions, '  shape', sub_m.shape)

u_m = Function(name='u_m', grid=grid_m, space_order=2)
vp_m = Function(name='vp_m', grid=sub_m, space_order=2)
print(Operator(Eq(u_m, vp_m)))

dimensions (xs, ys, z)   shape (7, 9, 13)


/* Devito generated code for Operator `Kernel` */

#define _POSIX_C_SOURCE 200809L
#define START(S) struct timeval start_ ## S , end_ ## S ; gettimeofday(&start_ ## S , NULL);
#define STOP(S,T) gettimeofday(&end_ ## S, NULL); T->S += (double)(end_ ## S .tv_sec-start_ ## S.tv_sec)+(double)(end_ ## S .tv_usec-start_ ## S .tv_usec)/1000000;

#include "stdlib.h"
#include "math.h"
#include "sys/time.h"
#include "xmmintrin.h"
#include "pmmintrin.h"

struct dataobj
{
  void *restrict data;
  int * size;
  unsigned long nbytes;
  unsigned long * npsize;
  unsigned long * dsize;
  int * hsize;
  int * hofs;
  int * oofs;
  void * dmap;
} ;

struct profiler
{
  double section0;
} ;


int Kernel(struct dataobj *restrict u_m_vec, struct dataobj *restrict vp_m_vec, const int x_M, const int x_m, const int y_M, const int y_m, const int z_M, const int z_m, struct profiler * timers)
{
  float w6c8dafcdi[6] __attribute__ ((aligned (64))) = {1.0F, 0.0F, 3.33333333333333e-1F, 6.66666666666667e-1F, 6.66666

One weight table per axis that needs interpolation (`w0` for `x`, 2 phases; `w1` for
`y`, 3 phases) and none for the `z` axis of factor 1. All three loops are plain and
unit-stride.

### Which direction interpolates

The factor decides whether a *strided write* is possible; it does not decide whether a
*read* interpolates. Writing the SubGrid visits only its own points, so with an integer
factor `Eq(vp, u)` is a plain strided copy with no weights at all:

In [45]:
# NBVAL_IGNORE_OUTPUT
sub_int = SubGrid(grid_i, factor=2, name='k')
print('interpolated:', sub_int.is_interpolated, ' dimensions:', sub_int.dimensions)

vp_int = Function(name='vp_int', grid=sub_int, space_order=2)
print(Operator(Eq(vp_int, u_i)))

interpolated: False  dimensions: (xk, yk)


/* Devito generated code for Operator `Kernel` */

#define _POSIX_C_SOURCE 200809L
#define START(S) struct timeval start_ ## S , end_ ## S ; gettimeofday(&start_ ## S , NULL);
#define STOP(S,T) gettimeofday(&end_ ## S, NULL); T->S += (double)(end_ ## S .tv_sec-start_ ## S.tv_sec)+(double)(end_ ## S .tv_usec-start_ ## S .tv_usec)/1000000;

#include "stdlib.h"
#include "math.h"
#include "sys/time.h"
#include "xmmintrin.h"
#include "pmmintrin.h"

struct dataobj
{
  void *restrict data;
  int * size;
  unsigned long nbytes;
  unsigned long * npsize;
  unsigned long * dsize;
  int * hsize;
  int * hofs;
  int * oofs;
  void * dmap;
} ;

struct profiler
{
  double section0;
} ;


int Kernel(struct dataobj *restrict u_i_vec, struct dataobj *restrict vp_int_vec, const int xk_M, const int xk_m, const int xkf, const int yk_M, const int yk_m, const int ykf, struct profiler * timers)
{
  float (*restrict u_i)[u_i_vec->size[1]] __attribute__ ((aligned (64))) = (float (*)[u_i_vec->size[1]]) u_i_vec-

Reading it back onto the parent grid is the other direction: `Eq(u, vp)` is evaluated
at *every* parent point, and only every 2nd of those coincides with a `sub_int` point.
The ones in between are interpolated -- so an integer factor does get weights here.

In [46]:
# NBVAL_IGNORE_OUTPUT
print(Operator(Eq(u_i, vp_int)))

/* Devito generated code for Operator `Kernel` */

#define _POSIX_C_SOURCE 200809L
#define START(S) struct timeval start_ ## S , end_ ## S ; gettimeofday(&start_ ## S , NULL);
#define STOP(S,T) gettimeofday(&end_ ## S, NULL); T->S += (double)(end_ ## S .tv_sec-start_ ## S.tv_sec)+(double)(end_ ## S .tv_usec-start_ ## S .tv_usec)/1000000;

#include "stdlib.h"
#include "math.h"
#include "sys/time.h"
#include "xmmintrin.h"
#include "pmmintrin.h"

struct dataobj
{
  void *restrict data;
  int * size;
  unsigned long nbytes;
  unsigned long * npsize;
  unsigned long * dsize;
  int * hsize;
  int * hofs;
  int * oofs;
  void * dmap;
} ;

struct profiler
{
  double section0;
} ;


int Kernel(struct dataobj *restrict u_i_vec, struct dataobj *restrict vp_int_vec, const int x_M, const int x_m, const int y_M, const int y_m, struct profiler * timers)
{
  float w50937fb0i[4] __attribute__ ((aligned (64))) = {1.0F, 0.0F, 5.0e-1F, 5.0e-1F};

  float (*restrict u_i)[u_i_vec->size[1]] __attribute__ ((a

`w0 = {1.0, 0.0, 0.5, 0.5}`: the even parent points take a single tap of weight 1
(they sit exactly on a SubGrid point), the odd ones average their two neighbours. The
loops are over the parent's `x`, `y`, so every point of `u` is written.

### Scattering instead: `subdomain=`

Interpolating onto every parent point is the default because it follows the LHS. When
you want the other behaviour -- visit the subgrid's points alone and leave the rest of
the parent untouched -- make the subgrid the equation's iteration domain:

In [47]:
# NBVAL_IGNORE_OUTPUT
print(Operator(Eq(u_i, vp_int, subdomain=sub_int)))

/* Devito generated code for Operator `Kernel` */

#define _POSIX_C_SOURCE 200809L
#define START(S) struct timeval start_ ## S , end_ ## S ; gettimeofday(&start_ ## S , NULL);
#define STOP(S,T) gettimeofday(&end_ ## S, NULL); T->S += (double)(end_ ## S .tv_sec-start_ ## S.tv_sec)+(double)(end_ ## S .tv_usec-start_ ## S .tv_usec)/1000000;

#include "stdlib.h"
#include "math.h"
#include "sys/time.h"
#include "xmmintrin.h"
#include "pmmintrin.h"

struct dataobj
{
  void *restrict data;
  int * size;
  unsigned long nbytes;
  unsigned long * npsize;
  unsigned long * dsize;
  int * hsize;
  int * hofs;
  int * oofs;
  void * dmap;
} ;

struct profiler
{
  double section0;
} ;


int Kernel(struct dataobj *restrict u_i_vec, struct dataobj *restrict vp_int_vec, const int xk_M, const int xk_m, const int xkf, const int yk_M, const int yk_m, const int ykf, struct profiler * timers)
{
  float (*restrict u_i)[u_i_vec->size[1]] __attribute__ ((aligned (64))) = (float (*)[u_i_vec->size[1]]) u_i_vec-

One strided loop over `xk`, `yk` writing `u[xk*xkf]`, and no weights at all: this is
Part 1's lowering, reached through the same equation. The two differ only in the
iteration domain.

An *interpolated* subgrid has no points on the parent grid, so there is nothing to
scatter onto and asking for it is an error rather than a silent fallback:

In [48]:
try:
    Eq(u_i, vp_i, subdomain=sub_i)
except ValueError as e:
    print(e)

Cannot iterate the interpolated SubGrid `SubGrid[shape=(9, 9), factor=(1.5, 1.5)]`: with factor (1.5, 1.5) its points fall *between* the parent Grid's, so there are no shared points to iterate. Drop `subdomain=` to interpolate onto every parent point instead


In [49]:
vp_int.data[:] = 3.*np.arange(sub_int.shape[0])[:, None]
u_i.data[:] = -1.

Operator(Eq(u_i, vp_int)).apply()

idx = np.arange(grid_i.shape[0])
inside = idx <= (sub_int.shape[0] - 1)*2 - 1
print('any point left unwritten:', bool(np.any(np.asarray(u_i.data) == -1.)))
print('max error vs 3*(x/2) :',
      np.abs(np.asarray(u_i.data)[inside, 0] - 3.*(idx[inside]/2.)).max())

Operator `Kernel` ran in 0.01 s


any point left unwritten: False
max error vs 3*(x/2) : 0.0


## Interpolation width: `interp_order`

The stencil width is **not** a subgrid-specific setting. A subgrid read is an
ordinary finite-difference relocation -- the same one staggering uses -- so the
Function's own `interp_order` governs it, just as it does everywhere else in Devito:

In [50]:
# NBVAL_IGNORE_OUTPUT
import re

grid_s = Grid(shape=(33,), extent=(32.,))
sub_s = SubGrid(grid_s, factor=1.5)
u_s = Function(name='u_s', grid=grid_s, space_order=8)

for io in (2, 4, 8):
    vp_s = Function(name=f'vp{io}', grid=sub_s, space_order=8, interp_order=io)
    src = str(Operator(Eq(u_s, vp_s)))
    body = [l for l in src.splitlines() if '] =' in l and 'u_s[' in l][0]
    pattern = r'vp' + str(io) + r'\[([^\]]+)\]'
    taps = sorted(set(re.findall(pattern, body)))
    print(f'interp_order={io}: {len(taps)} taps')

interp_order=2: 2 taps


interp_order=4: 4 taps


interp_order=8: 8 taps


### Accuracy

Which scheme is best depends on how well the subgrid resolves the field. On a sine
wave, sampled at a few different wavelengths:

In [51]:
# NBVAL_IGNORE_OUTPUT
from devito.logger import set_log_level

set_log_level('WARNING')     # one 'ran in' line per apply otherwise

n_a, fac_a = 129, 1.5
grid_a = Grid(shape=(n_a, n_a), extent=(float(n_a - 1),)*2)
px_a, py_a = np.meshgrid(np.arange(n_a), np.arange(n_a), indexing='ij')
k = 2*np.pi*8/(n_a - 1)          # ~11 subgrid points per cycle

for io in (2, 4, 8):
    sub_a = SubGrid(grid_a, factor=fac_a, name=f's{io}')
    u_a = Function(name=f'ua{io}', grid=grid_a, space_order=8)
    vp_a = Function(name=f'va{io}', grid=sub_a, space_order=8, interp_order=io)

    i_a, j_a = np.meshgrid(*[np.arange(s) for s in sub_a.shape], indexing='ij')
    vp_a.data[:] = np.sin(k*i_a*fac_a)*np.cos(k*j_a*fac_a)
    u_a.data[:] = 0.

    Operator(Eq(u_a, vp_a)).apply()

    pad = io//2 + 2
    mask_a = np.ones_like(px_a, dtype=bool)
    for q, c in enumerate((px_a, py_a)):
        mask_a &= (c >= pad) & (c <= int((sub_a.shape[q] - 1)*fac_a) - pad)
    exact_a = np.sin(k*px_a)*np.cos(k*py_a)
    err = np.abs(np.asarray(u_a.data)[mask_a] - exact_a[mask_a]).max()
    print(f'interp_order={io}: max error {err:.2e}')

set_log_level('INFO')

interp_order=2: max error 7.49e-02


interp_order=4: max error 4.79e-03


interp_order=8: max error 2.48e-05


Accuracy improves with the width, as it does for any FD stencil. The weights are the
proper Lagrange coefficients, so **every** width reproduces a linear field exactly.

The cost is `interp_order**ndim` terms per point, so the width is a real tradeoff in 3D
-- `interp_order=2` is the default for that reason.

`interp_order <= space_order` is already a `Function` invariant, and it is also what
keeps the taps inside the halo under MPI. There is no separate halo bookkeeping for
subgrid reads.

### Windowed sinc

One alternative is available, for the case FD weights do not cover -- Kaiser windowed
sinc, after Hicks (2002):

```python
sub = SubGrid(grid, factor=1.5, interpolation='sinc')
```

`interp_order` still sets the number of taps. Note that a *narrow* windowed sinc is a
poor interpolant: at `interp_order=2` it does not even match a linear field's first
moment. It only becomes competitive at 8 taps or more, where it trades exactness on
polynomials for better behaviour on band-limited data.

## Under MPI

Interpolated reads work under domain decomposition, with one requirement to be aware
of.

The parent `Grid` and the `SubGrid` are decomposed **independently** -- their split
points are computed separately over different shapes -- so a rank's parent points
generally need subgrid points that its *neighbour* owns. Devito handles this by
emitting a halo exchange for the subgrid Function, even though the equation never
writes it: `vp.data[:] = ...` fills only the local domain, so the halo would otherwise
be stale.

The second interpolation tap reads `base + 1`, which lands in that halo. **A subgrid
Function read under MPI therefore needs `space_order >= 1`**; with `space_order=0`
there is no halo to receive the neighbour's point, and rather than read out of bounds
Devito raises:

```python
vp = Function(name='vp', grid=sub, space_order=0)
Operator(Eq(u, vp))
# ValueError: `vp` is defined on the interpolated SubGrid ... with no halo along
# `xs`, so interpolating it onto its parent Grid under MPI would read out of
# bounds. Give `vp` a `space_order` of at least 1
```

One layer is enough for linear interpolation. Serial runs are unaffected -- with no
decomposition there is no neighbour to consult, and `space_order=0` is fine.

There is a second subtlety worth knowing about, because it is invisible in the
generated code until you look for it. The base index `(x*q)//p` uses integer division,
which is **non-linear**: it has to be evaluated on the *global* index and converted
back, not on the rank-local one. Devito passes the two per-rank offsets in as
`oi_x`/`oi_xs` and the stencil reads

```c
vp[-oi_xs + ((2*x + 2*oi_x) / 3) + 2]
```

Without decomposition both are literal zero, so the serial loop keeps the simpler form
shown earlier and pays nothing for this.

## Summary

`SubGrid(parent, factor)` builds a grid from a parent `Grid`. It is a `Grid` subclass,
so Functions, TimeFunctions, Operators and finite differences all work as usual.

```python
sub = SubGrid(grid, factor=2)             # or 1.5, or (2, 1.5), or 0.5
vp  = Function(name='vp', grid=sub)       # shape and dimensions derived

Operator(Eq(vp, u)).apply()               # write the subgrid: its points only
Operator(Eq(u, vp*u)).apply()             # read it: vp evaluated at u's points
Operator(Eq(u, vp, subdomain=sub)).apply()  # scatter: subgrid points of u only
```

The factor decides the *shape*, and whether the subgrid's points coincide with the
parent's at all:

| | integer factor | non-integer factor |
|---|---|---|
| `is_interpolated` | `False` | `True` |
| `shape` | `ceil(n/factor)` | `floor((n-1)/factor) + 1` |
| `dimensions` | `ConditionalDimension`s on the parent's | own `SpaceDimension`s |
| `Eq(vp, u)` (write) | strided copy, no weights | not a copy |
| `subdomain=sub` (scatter) | strided scatter | `ValueError` -- no shared points |

The *direction* decides whether interpolation happens, independently of the factor:

- **writing** the subgrid visits only its own points, so an integer factor needs no
  interpolation at all;
- **reading** it onto the parent grid computes every parent point by default, so the
  points between two subgrid points are interpolated -- including when the factor is an
  integer. Pass `subdomain=sub` to scatter onto the subgrid's points instead.

Per-axis factors are independent, and `1` passes the parent's own Dimension through
unchanged. Common to both regimes: `spacing` is `factor * parent.spacing`; `extent`,
`origin`, `dtype` and `time_dim` are inherited; `parent` and `factor` are the
attributes `SubGrid` adds.

Use a `ConditionalDimension` directly for subsampling in **time**, or to combine a
factor with an extra `condition` -- `SubGrid` covers neither.

Under the hood, a strided write or scatter iterates the subgrid with a unit stride and
rescales parent accesses, falling back to a pointwise `if (x%factor == 0)` guard in the
four cases above. An interpolating read becomes a plain stencil whose weights come from
a compile-time table indexed by the phase `x % p`, so the loops stay unit-stride and
branch-free. Linear interpolation is used today; it is exact on a linear field.

The stencil width follows the Function's own `interp_order`, exactly as it does for a
staggered relocation -- a subgrid read *is* an FD relocation. Wider is more accurate,
at `interp_order**ndim` terms per point, and every width is exact on a linear field.
`interpolation='sinc'` swaps the FD weights for Kaiser windowed ones at the same width.

Everything works under MPI, with no extra bookkeeping: `interp_order <= space_order` is
already a `Function` invariant, and it is what keeps the taps inside the halo.
